# 07 — Experiment Tracking & Iteration

**Purpose:** Compare experiment runs, validate that new features improve the model,
and maintain a history of all iterations.

This notebook is the control room for iteration:
- See all past experiments at a glance
- Compare any two runs (features added, metric delta)
- Decide whether to promote a new feature set
- Get a clear verdict: IMPROVED / DEGRADED / NO CHANGE

---
**Inputs:** `experiments/*.json`  
**Outputs:** `reports/experiment_history.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiment_tracker import (
    load_all_experiments, compare_experiments, get_best_run,
    diff_experiments, print_diff,
    plot_experiment_history, plot_feature_count_vs_metric,
    log_experiment,
)
from src.data_utils import REPORTS_DIR

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)
print('Ready.')

## 1. All Experiments

In [ ]:
all_exp = load_all_experiments()

if all_exp.empty:
    print('No experiments logged yet. Run notebook 06 first.')
else:
    print(f'{len(all_exp)} experiments found.\n')
    display(all_exp)

## 2. Rank by C-index

In [ ]:
ranked = compare_experiments(metric='c_index', higher_is_better=True)
if not ranked.empty:
    display(ranked[['c_index', 'ibs', 'n_features', 'notes']].round(4))

## 3. Best Run

In [ ]:
best = get_best_run(metric='c_index')
if best:
    print('Best run:')
    print(f'  Run name  : {best["run_name"]}')
    print(f'  C-index   : {best["metrics"]["c_index"]:.4f}')
    print(f'  IBS       : {best["metrics"]["ibs"]:.4f}')
    print(f'  Features  : {best["features_used"]}')
    print(f'  Notes     : {best["notes"]}')

## 4. Experiment History Chart

In [ ]:
plot_experiment_history(metric='c_index')
plot_experiment_history(metric='ibs')

In [ ]:
plot_feature_count_vs_metric(metric='c_index')

## 5. Comparing Two Runs

Change the run names below to compare any two experiments.

In [ ]:
# CHANGE THESE to the runs you want to compare
RUN_A = 'v1_cox_ph'    # baseline
RUN_B = 'v1_rsf'       # comparison

if not all_exp.empty:
    available = all_exp.index.tolist()
    if RUN_A in available and RUN_B in available:
        diff = diff_experiments(RUN_A, RUN_B)
        print_diff(diff)
    else:
        print(f'One or both runs not found. Available: {available}')
else:
    print('No experiments to compare yet.')

## 6. Simulated New Feature Iteration

This cell shows exactly what you'd do when adding a new feature:

1. Add feature to `config/features.yaml`
2. Implement in `src/feature_engineering.py`
3. Re-run notebooks 03 → 04 → 05 → 06
4. Come back here to compare runs

In [ ]:
# Simulate logging a hypothetical improved run (for demo purposes)
import json
from src.data_utils import DATA_PROCESSED

sel_path = DATA_PROCESSED / 'selected_features.json'
if sel_path.exists():
    current_features = json.load(open(sel_path))['selected_features']
    
    # Simulate adding a new derived feature
    hypothetical_features = current_features + ['new_feature_example']
    
    log_experiment(
        run_name='v2_with_new_feature_example',
        features_used=hypothetical_features,
        metrics={'c_index': 0.7420, 'ibs': 0.1580},   # replace with real metrics
        params={'model': 'rsf'},
        notes='Added new_feature_example — testing if it improves discrimination',
    )
    
    print('Hypothetical run logged. Now compare:')
    diff = diff_experiments('v1_rsf', 'v2_with_new_feature_example')
    print_diff(diff)

## 7. Save Full Experiment History

In [ ]:
all_exp_updated = load_all_experiments()

if not all_exp_updated.empty:
    path = REPORTS_DIR / 'experiment_history.csv'
    all_exp_updated.to_csv(path)
    print(f'Experiment history saved → {path}')
    display(all_exp_updated.round(4))

---
## Quick-Iteration Cheat Sheet

```
┌─────────────────────────────────────────────────────────────┐
│  Add a new feature (< 2 min)                                │
│  1. config/features.yaml → derived_features: [new_feat]    │
│  2. src/feature_engineering.py → _compute_derived_features  │
│  3. Re-run notebooks 03 → 04 → 05 → 06 → 07                │
│                                                             │
│  Re-run full pipeline (< 5 min on 2k rows)                  │
│  $ jupyter nbconvert --to notebook --execute                │
│      notebooks/03_feature_engineering.ipynb                 │
│      notebooks/04_feature_selection.ipynb                   │
│      notebooks/05_modeling_survival.ipynb                   │
│      notebooks/06_model_evaluation.ipynb                    │
│                                                             │
│  Compare before/after                                       │
│  diff = diff_experiments('v1_rsf', 'v2_rsf_new_feat')      │
│  print_diff(diff)                                           │
└─────────────────────────────────────────────────────────────┘
```